In [20]:
import json
import time

from openai import OpenAI
import dotenv
import os
dotenv.load_dotenv()
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

with open("dataset.txt", "r") as f:
  prompts = f.read().split("\n-\n")

In [21]:
prompts = [p.strip() for p in prompts if p.strip() != ""]
len(prompts)

228

In [22]:
import textwrap
def pprint(text):
    print(textwrap.fill(str(text), 100))

In [23]:
models = ["google/gemini-2.0-flash-001",
          "google/gemini-2.5-flash",
          "google/gemini-3-flash-preview",
          "openai/gpt-5.3-chat",
          "openai/gpt-5-chat",
          "openai/gpt-4.1",
          "openai/gpt-4o-2024-11-20",
          "openai/gpt-4o-2024-05-13",
          "openai/gpt-4-turbo",
          "openai/gpt-3.5-turbo",
          "anthropic/claude-sonnet-4.6",
          "x-ai/grok-4.20", 
          "anthropic/claude-3.5-haiku",
          "anthropic/claude-3.7-sonnet",
          "anthropic/claude-sonnet-4",
          "qwen/qwen3.6-plus",
          ]

In [24]:
ds = []
for prompt in prompts:
  if prompt == "":
    continue

  for model in models: 
    ds.append({
        "prompt": prompt,
        "model": model,
    })

In [25]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def _call_api(sample):
    return client.chat.completions.create(
        model=sample["model"],
        messages=[
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": "You are a helpful assistant."}
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["prompt"]}
                ]
            }
        ],
        extra_body={
            "reasoning": {
                "effort": "none"
            }
        } 
    )

def eval(sample, max_retries=5):
    retries = 0
        
    while True:
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(_call_api, sample)
            try:
                completion = future.result(timeout=60)
                if completion.choices[0].message.content:
                    sample["response"] = completion.choices[0].message.content
                    return sample
            except FuturesTimeout:
                retries += 1
                if retries >= max_retries:
                    raise TimeoutError("Max retries exceeded")
                continue
            except Exception as e:
                print(e)
                continue


In [26]:
len(ds)

3648

In [27]:
results = {}

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, FIRST_COMPLETED, wait
from tqdm import tqdm

executor = ThreadPoolExecutor(max_workers=30)
futures = {executor.submit(eval, results[idx]): idx for idx in range(len(ds)) if idx not in results}

try:
    pending = set(futures)
    with tqdm(total=len(futures)) as pbar:
        while pending:
            done, pending = wait(pending, timeout=0.5, return_when=FIRST_COMPLETED)
            for future in done:
                idx = futures[future]
                results[idx] = future.result()
                pbar.update(1)
except KeyboardInterrupt:
    print("interrupted, cancelling...")
    for f in futures:
        f.cancel()
    executor.shutdown(wait=False, cancel_futures=True)
    raise
else:
    executor.shutdown()


In [29]:
futures

{}

0it [00:00, ?it/s]
